# 🌍 Module 1: Spatial-Temporal ETL, Population Rollup & Lag Feature Engineering
## **Project CCHAIN: Urban Heat Stress, Air Pollution & Cardiorespiratory Mortality Modeling Engine**
---
**Authors:** Environmental Health Data Science & Systems Architecture Team  
**Scope:** 879 Barangays $\to$ 12 Major Philippine Metropolitan Centers (2006–2021)  

### 🎯 Operational Objectives
1. **Ingest Raw CCHAIN Multi-Modal Datasets:** Load 6.42M daily atmosphere and 6.42M daily air quality records, WorldPop rasters, ESA WorldCover, Relative Wealth Index (RWI), and PSA cause-specific weekly mortality counts.
2. **Population-Weighted Spatial Rollup:** Aggregate barangay metrics ($adm4$) to city administrative scale ($adm3$) using annual WorldPop weights ($W_{b,y} = \text{Pop}_{b,y} / \text{CityPop}_{c,y}$).
3. **Demographic CAGR Extrapolation:** Extrapolate host population estimates for 2021–2022 using 10-year compound annual growth rates.
4. **Zero-Padding & Selection Bias Elimination:** Construct a full Cartesian product grid ($12 \text{ Cities} \times 834 \text{ ISO Weeks} \times 4 \text{ Causes}$) and zero-fill missing mortality reports.
5. **Distributed Lag & Compound Feature Engineering:** Generate 0–14 day lag structures, rolling EWMAs, Urban Heat Island (UHI) proxies, and compound hazard synergy terms ($HI_{95\text{th}} \times PM_{2.5\text{mean}}$).
6. **Rate Normalization & Master Export:** Standardize weekly deaths into rates per 100,000 population and export to `data/processed_cchain_master.csv`.

### 📦 Step 1: Environment Setup & Library Imports
Importing necessary data processing, spatial indexing, and visualization libraries.

In [ ]:
import os
import sys
import logging
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set high-resolution plotting
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 150

# Configure directory paths
RAW_DIR = Path('../cchain_raw') if Path('../cchain_raw').exists() else Path('data/cchain_raw')
OUTPUT_DIR = Path('data')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Raw Data Directory: {RAW_DIR.resolve()}')
print(f'Processed Output Directory: {OUTPUT_DIR.resolve()}')

### 🗺️ Step 2: Spatial Reference & Administrative Hierarchy
Inspect the spatial mapping across Regions ($adm1$), Provinces ($adm2$), Cities ($adm3$), and Barangays ($adm4$).

In [ ]:
df_loc = pd.read_csv(RAW_DIR / 'location.csv')
print(f'Total Barangays: {len(df_loc)}')
print(f'Total Cities: {df_loc["adm3_pcode"].nunique()}')

city_summary = df_loc.groupby(['adm3_pcode', 'adm3_en', 'adm2_en', 'adm1_en'])['adm4_pcode'].count().reset_index()
city_summary.columns = ['City PCode', 'City Name', 'Province', 'Region', 'Barangay Count']
display(city_summary)

### 👥 Step 3: WorldPop Host Density & CAGR Population Extrapolation (2000–2022)
To prevent exposure misclassification, daily environmental metrics must be weighted by host population:
$$W_{b, y} = \frac{\text{Pop}_{b, y}}{\sum_{i \in c} \text{Pop}_{i, y}}$$
For years 2021–2022, we apply localized 10-year historical growth trends:
$$r_b = \left( \frac{\text{Pop}_{b, 2020}}{\text{Pop}_{b, 2010} + \epsilon} \right)^{\frac{1}{10}} - 1, \quad \text{Pop}_{b, y} = \text{Pop}_{b, 2020} \times (1 + r_b)^{y - 2020}$$

In [ ]:
from src.data_processing import CCHAINDataPipeline

pipeline = CCHAINDataPipeline(raw_data_dir=RAW_DIR, output_dir=OUTPUT_DIR)
df_weights, df_city_pop = pipeline.build_population_weights(df_loc)

# Inspect sample city population trajectories
piv_pop = df_city_pop.pivot(index='adm3_pcode', columns='year', values='city_pop_total')
sample_years = [2006, 2010, 2015, 2020, 2021, 2022]
display(piv_pop[[y for y in sample_years if y in piv_pop.columns]].head(6))

### 🏙️ Step 4: Built Environment (ESA WorldCover) & Socioeconomic Vulnerability (RWI)
We compute:
1. **Urban Heat Island Proxy Ratio:**
   $$\text{UHI\_Proxy}_c = \frac{\text{pct\_builtup}_c}{\text{pct\_tree\_cover}_c + 0.01}$$
2. **Socio-Environmental Vulnerability Index (SEVI):**
   $$\text{SEVI}_{c, y} = (1 - \text{RWI}_{c, y}) \times \ln(1 + \text{CityPopDensity}_{c, y})$$

In [ ]:
df_static = pipeline.build_static_and_socioeconomic_features(df_weights, df_city_pop)
display(df_static[df_static['year'] == 2020].head(6))

### 🌡️ Step 5: Streaming Population-Weighted Daily Spatial Rollup
Processing 6.42M daily atmosphere and 6.42M daily air quality rows in chunks to create daily city-level exposure time series:
$$\bar{X}_{c, t} = \sum_{b \in c} W_{b, y(t)} \cdot X_{b, t}$$

In [ ]:
df_daily_env = pipeline.rollup_daily_environmental_data(df_weights)
print(f'Aggregated Daily Environmental Records: {len(df_daily_env)}')
display(df_daily_env.head(5))

### ⏱️ Step 6: ISO Weekly Aggregation, Distributed Lags (0–14 Days) & Compound Hazards
Resample daily records into ISO weekly metrics (mean, max, 95th percentile, hot days $\ge 37^\circ\text{C}$) and engineer lag polynomials:
- `heat_index_lag1`, `heat_index_lag2`, `heat_index_roll2w_mean`, `heat_index_ewma`
- `pm25_lag1`, `pm25_lag2`, `pm25_roll2w_mean`, `pm25_ewma`
- Compound Interaction: $\text{CompoundRisk} = \text{HeatIndex}_{95\text{th}} \times \text{PM2.5}_{\text{mean}}$

In [ ]:
df_weekly_env = pipeline.aggregate_weekly_features_and_lags(df_daily_env)
print(f'Weekly Environmental Feature Table Shape: {df_weekly_env.shape}')
display(df_weekly_env[['adm3_pcode', 'week_start_date', 'heat_index_mean', 'heat_index_p95', 'pm25_mean', 'compound_risk_hi95_pm25']].head())

### 🏥 Step 7: Mortality Ingestion, Zero-Padding & Rate Standardization
Reindex against continuous Cartesian Grid ($12 \text{ Cities} \times 834 \text{ Weeks}$) and standardize rates per 100,000 host population:
$$\text{Mortality Rate}_{c, w, d} = \left( \frac{\text{Death Total}_{c, w, d}}{\text{CityPop}_{c, y(w)}} \right) \times 100,000$$

In [ ]:
df_mort = pipeline.process_mortality_and_grid(df_city_pop)
print(f'Mortality Zero-Padded Grid Records: {len(df_mort)}')
display(df_mort.head())

### 🚀 Step 8: Master Fusion, Philippine Seasonality & Export
Fusing all feature layers and exporting the complete dataset to `data/processed_cchain_master.csv`.

In [ ]:
df_master = pipeline.execute_pipeline()
print(f'Master Modeling Dataset Dimensions: {df_master.shape}')
print(f'Date Range: {df_master["week_start_date"].min().date()} to {df_master["week_start_date"].max().date()}')
display(df_master.head())

### 📊 Step 9: Exploratory Environmental Health Visualizations

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Heat Index vs PM2.5 Compound Distribution by Season
sns.scatterplot(
    data=df_master, x='heat_index_mean', y='pm25_mean', hue='ph_season',
    palette={'Hot-Dry': '#e41a1c', 'Wet': '#377eb8', 'Cool-Dry': '#4daf4a'}, alpha=0.6, ax=axes[0, 0]
)
axes[0, 0].set_title('A. Compound Thermal-Pollution Distribution by Climate Season', fontsize=12, fontweight='bold')
axes[0, 0].set_xlabel('Weekly Mean Heat Index (°C)')
axes[0, 0].set_ylabel('Weekly Mean PM2.5 (µg/m³)')

# 2. Total Cardiorespiratory Mortality Rates Across Cities
sns.boxplot(data=df_master, x='adm3_en', y='rate_cardiorespiratory_per_100k', palette='Set3', ax=axes[0, 1])
axes[0, 1].set_title('B. Cardiorespiratory Mortality Rate Distributions Across 12 Cities', fontsize=12, fontweight='bold')
axes[0, 1].tick_params(axis='x', rotation=45)
axes[0, 1].set_xlabel('')
axes[0, 1].set_ylabel('Rate / 100,000 Pop')

# 3. Urban Heat Island (UHI) Proxy Ratio Ranking
uhi_df = df_master[['adm3_en', 'uhi_proxy_ratio']].drop_duplicates().sort_values('uhi_proxy_ratio', ascending=False)
sns.barplot(data=uhi_df, x='adm3_en', y='uhi_proxy_ratio', palette='viridis', ax=axes[1, 0])
axes[1, 0].set_title('C. Urban Heat Island (UHI) Built-to-Green Ratio by City', fontsize=12, fontweight='bold')
axes[1, 0].tick_params(axis='x', rotation=45)
axes[1, 0].set_xlabel('')
axes[1, 0].set_ylabel('Built-Up / (Tree Cover + 0.01)')

# 4. Longitudinal Time Series (Mandaluyong vs Davao)
subset_cities = df_master[df_master['adm3_en'].isin(['City of Mandaluyong', 'Davao City'])]
sns.lineplot(data=subset_cities, x='week_start_date', y='rate_cardiorespiratory_per_100k', hue='adm3_en', ax=axes[1, 1])
axes[1, 1].set_title('D. Longitudinal Cardiorespiratory Rate Series (2006–2021)', fontsize=12, fontweight='bold')
axes[1, 1].set_xlabel('Calendar Year')
axes[1, 1].set_ylabel('Rate / 100k')

plt.tight_layout()
plt.show()